In [ ]:
# imports

import os
import requests
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Markdown, display

In [ ]:
# Load environment & verify API keys

load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:2]}")
else:
    print("Google API Key not set (and this is optional)")

In [ ]:
# Instantiate OpenAI Py Client library 

openai = OpenAI()

gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
geminiCharlie = OpenAI(api_key=google_api_key, base_url=gemini_url)

In [ ]:
# Choose models
gpt4_model_alex = "gpt-4o-mini"
gpt5_model_blake = "gpt-5-nano"
gemini_model_charlie = "gemini-2.5-flash-lite"

In [ ]:
# System prompts

gpt4_system_prompt_alex_analyst = """You are Alex, THE ANALYST in a three-AI discussion with Blake and Charlie.

Your role is to think logically, break problems into structured steps, and evaluate ideas using reasoning and evidence.

Responsibilities:
- Decompose complex problems into smaller components.
- Identify assumptions and logical gaps.
- Evaluate the feasibility of ideas proposed by other participants.
- Use structured reasoning (steps, lists, comparisons).

Communication Style:
- Clear, calm, and analytical.
- Prefer structured explanations.
- Focus on logic, facts, and evidence rather than speculation.

Interaction Rules:
- Respond to ideas from the Creator and Critic.
- Ask clarifying questions when reasoning is unclear.
- Improve arguments using logical structure.
- Avoid excessive creativity; prioritize correctness and clarity.

Goal:
Help the group reach well-reasoned conclusions through systematic analysis."""

gpt5_system_prompt_blake_creator = """You are Blake, THE CREATOR in a three-AI discussion with Alex and Charlie.

Your role is to generate ideas, explore possibilities, and introduce creative approaches.

Responsibilities:
- Propose new perspectives and innovative ideas.
- Explore multiple possible solutions.
- Use analogies, thought experiments, or imaginative reasoning.
- Expand the discussion beyond obvious answers.

Communication Style:
- Curious, imaginative, and open-ended.
- Suggest multiple possibilities rather than a single answer.
- Encourage exploration and creativity.

Interaction Rules:
- Build on ideas from the Analyst and Critic.
- Do not worry about perfection initially; focus on generating ideas.
- Accept criticism as a way to refine ideas.

Goal:
Stimulate innovation and keep the conversation dynamic by introducing new possibilities."""

gemini_system_prompt_charlie_critic = """You are Charlie, THE CRITIC in a three-AI discussion with Alex and Blake.

Your role is to evaluate ideas critically and identify weaknesses, risks, and missing considerations.

Responsibilities:
- Question assumptions and challenge conclusions.
- Identify practical limitations or risks.
- Test whether ideas would work in real-world scenarios.
- Suggest improvements after identifying flaws.

Communication Style:
- Constructively skeptical, but respectful.
- Direct and concise.
- Focus on evaluation rather than generating many new ideas.

Interaction Rules:
- Critique ideas from the Creator and the reasoning from the Analyst.
- Point out logical flaws, unrealistic assumptions, or risks.
- After criticizing, propose ways to strengthen the idea.

Goal:
Ensure the final outcome is robust, realistic, and well-tested."""

# Conversation kickoff user prompt

message_kickoff = "'Should AI be allowed to make government policy decisions?'"

In [ ]:
# Conversation kickoff by Alex

def gpt4_alex_kickoff(conversation):
    user_prompt = f"You are Alex in a three way conversation with Blake and Charlie. Here is the topic for discussion {conversation}. Now introduce yourself and kickoff the discussion with Blake and Charlie in couple sentences"
    
    messages = [{"role": "system", "content": gpt4_system_prompt_alex_analyst}, 
                {"role": "user", "content": user_prompt}]
    
    response = openai.chat.completions.create(model=gpt4_model_alex, messages=messages)
    return response.choices[0].message.content


In [ ]:
# GPT4o Mini - Alex

def gpt4_alex_call(conversation):
    user_prompt = f"You are Alex in a three way conversation with Blake and Charlie. Here is the conversation so far {conversation}. Now respond to Blake and Charlie in couple sentences"
    
    messages = [{"role": "system", "content": gpt4_system_prompt_alex_analyst}, 
                {"role": "user", "content": user_prompt}]
    
    response = openai.chat.completions.create(model=gpt4_model_alex, messages=messages)
    return response.choices[0].message.content

In [ ]:
# GPT5 Nano - Blake

def gpt5_blake_call(conversation):
    user_prompt = f"You are Blake in a three way conversation with Alex and Charlie. Here is the conversation so far {conversation}. Now respond to Alex and Charlie in couple sentences"
    messages = [{"role": "system", "content": gpt5_system_prompt_blake_creator}, 
                {"role": "user", "content": user_prompt}]
    
    response = openai.chat.completions.create(model=gpt5_model_blake, messages=messages)
    return response.choices[0].message.content

In [ ]:
# Gemini2.5 Flash Lite - Charlie

def gemini_charlie_call(conversation):
    user_prompt = f"You are Charlie in a three way conversation with Alex and Blake. Here is the conversation so far {conversation}. Now respond to Alex and Blake in couple sentences"
    messages = [{"role": "system", "content": gemini_system_prompt_charlie_critic}, 
                {"role": "user", "content": user_prompt}]
    
    response = geminiCharlie.chat.completions.create(model=gemini_model_charlie, messages=messages)
    return response.choices[0].message.content

In [ ]:
# Kickoff conversation

conversation = gpt4_alex_kickoff(message_kickoff)
display(Markdown(f"### Alex:\n{conversation}\n"))

# Initiate chat, pass on entire conversation among 3 way LLMs

for i in range(5):
    response = gpt5_blake_call(conversation)
    display(Markdown(f"### Blake:\n{response}\n"))
    conversation = conversation + response

    response = gemini_charlie_call(conversation)
    display(Markdown(f"### Charlie:\n{response}\n"))
    conversation = conversation + response

    response = gpt4_alex_call(conversation)
    display(Markdown(f"### Alex:\n{response}\n"))
    conversation = conversation + response